In [46]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import random
import string

tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2", torch_dtype="auto", device_map="auto")
model = AutoModelForCausalLM.from_pretrained("openai-community/gpt2", torch_dtype="auto", device_map="auto")

In [47]:
print(tokenizer.eos_token_id)
tokenizer.pad_token = tokenizer.eos_token
enc = tokenizer(
    ["Xin chào", "Hello world"],
    padding=True,
    return_attention_mask=True,
    return_tensors="pt"
)

50256


In [48]:
model_inputs = tokenizer(["The secret to baking a good cake is "], return_tensors="pt").to("cuda")
model_inputs

{'input_ids': tensor([[  464,  3200,   284, 16871,   257,   922, 12187,   318,   220]],
       device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')}

In [49]:
generated_ids = model.generate(**model_inputs, max_length=100)
tokenizer.batch_decode(generated_ids)[0]

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


"The secret to baking a good cake is \xa0to make sure that the cake is moist and not too moist. \xa0If you're baking a cake with a lot of cake, you'll want to make sure that the cake is moist and not too moist. \xa0If you're baking a cake with a lot of cake, you'll want to make sure that the cake is moist and not too moist. \xa0If you're baking a cake with a lot of cake, you'll want"

In [50]:
from datasets import load_dataset

In [51]:
dataset = load_dataset("rotten_tomatoes")
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
})

In [52]:
def tokenize_dataset(dataset):
    return tokenizer(dataset["text"])
dataset = dataset.map(tokenize_dataset, batched=True)

In [53]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 1066
    })
})

In [54]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="distilbert-rotten-tomatoes",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    push_to_hub=True,
    num_train_epochs=2,
)

In [55]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


In [56]:
examples = [
    "Hôm nay trời đẹp",
    "Tôi đang học NLP với Hugging Face"
]

tokenized_examples = [
    tokenizer(text, add_special_tokens=True, truncation=True, return_attention_mask=True)
    for text in examples
]
print(len(tokenized_examples[0]['attention_mask']))

print(len(tokenized_examples[1]['attention_mask']))

batch = data_collator(tokenized_examples)

print("input_ids:\n", batch["input_ids"])
print("attention_mask:\n", batch["attention_mask"])

16
21
input_ids:
 tensor([[   39, 27083,    76,   299,   323,   491,   157,   119,   251,    72,
         34754,   239,   157,   118,   117,    79, 50256, 50256, 50256, 50256,
         50256],
        [   51, 27083,    72, 34754,   239,   648,   289,   157,   119,   235,
            66,   399, 19930,   410,   157,   119,   249,    72, 12905,  2667,
         15399]])
attention_mask:
 tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])


In [59]:
from transformers import Trainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="distilbert-rotten-tomatoes",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    push_to_hub=True,
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
)

trainer.train()

/tmp/ipykernel_857359/1594290665.py:12: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


ValueError: Expected input batch_size (352) to match target batch_size (8).